# LSTM, GRU: NER


In [ ]:
import collections
import gzip
import types
from typing import Optional

import evaluate
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import wandb
from matplotlib import pyplot as plt
from nltk import word_tokenize
from torch.utils.data import DataLoader
from transformers import Trainer, TrainingArguments
from transformers.modeling_outputs import TokenClassifierOutput

from helpers.config import DATA_DIR, PROCESSED_DIR

NER_DIR = DATA_DIR / "ukr-synth" / "data"
OUT_DIR = PROCESSED_DIR / "lecture" / "lstm_gru_ner"
assert NER_DIR.exists(), f"{NER_DIR} missing: download via hf CLI"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Data


In [ ]:
def read_conll_file(filepath, nrows=10):
    sentences = []
    current_sentence = []
    sentence_counts = 0
    with gzip.open(filepath, "rt", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            # A blank line indicates the end of a sentence
            if not line:
                if current_sentence:
                    sentences.append(current_sentence)
                    sentence_counts += 1
                    current_sentence = []
            else:
                # Split the line into its fields (e.g., token, POS, label, etc.)
                parts = line.split()
                current_sentence.append(parts)
            if sentence_counts >= nrows:
                break
        # If the file doesn't end with a newline, add the last sentence.
        if current_sentence:
            sentences.append(current_sentence)

    return sentences


train_data = read_conll_file(NER_DIR / "train.conllu.gz", nrows=200_000)
valid_data = read_conll_file(NER_DIR / "dev.conllu.gz", nrows=10_000)
print("Number of train sentences:", len(train_data))
print("Number of valid sentences:", len(valid_data))
print("Sentence example:", train_data[1])

In [ ]:
columns = [
    "ID",
    "FORM",
    "LEMMA",
    "UPOS",
    "XPOS",
    "FEATS",
    "HEAD",
    "DEPREL",
    "DEPS",
    "MISC",
]

sentence = pd.DataFrame(train_data[1][2:], columns=columns)
sentence

In [ ]:
def get_label_distribution(sentences, label_index=9):
    label_counter = collections.Counter()
    for sentence in sentences:
        for token in sentence:
            if token[0].startswith("#") or len(token) != 10:
                continue
            label = token[label_index]
            label_counter[label] += 1
    return label_counter


label_dist = get_label_distribution(train_data)

print("Unique labels:")
print(list(label_dist.keys()))

print("\nLabel distribution:")
for label, count in label_dist.items():
    print(f"{label}: {count}")

## Vocab


In [ ]:
class Vocab(dict):
    """token -> id, specials first, then tokens by frequency; unknown tokens map to <unk>."""

    def __init__(self, token_lists, specials):
        counts = collections.Counter(tok for toks in token_lists for tok in toks)
        rest = sorted(
            (t for t in counts if t not in specials), key=lambda t: (-counts[t], t)
        )
        super().__init__((tok, i) for i, tok in enumerate([*specials, *rest]))

    def __missing__(self, token):
        return self["<unk>"]

    def __call__(self, tokens):
        return [self[tok] for tok in tokens]

    lookup_indices = __call__


def yield_tokens(data):
    for sentence in data:
        tokens = [row[1] for row in sentence if not row[0].startswith("#")]
        yield tokens


vocab = Vocab(yield_tokens(train_data + valid_data), specials=["<unk>", "<pad>"])


# Define a tokenizer that extracts tokens from a given sentence (ignoring metadata rows)
def simple_tokenizer(sentence):
    return [row[1] for row in sentence if not row[0].startswith("#")]


# Example usage:
tokens = simple_tokenizer(train_data[1])
token_indices = [vocab[token] for token in tokens]

print("Tokens:", tokens)
print("Token indices:", token_indices)

In [ ]:
len(vocab)

In [ ]:
vocab.lookup_indices(["поки", "що", "покищо", "покищо#", "<unk>", "<pad>"])

In [ ]:
token_freqs = collections.Counter(
    token for tokens in yield_tokens(train_data) for token in tokens
)

# Get the 30 most common tokens
top30 = token_freqs.most_common(30)
top30

## Dataset


In [ ]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, conll_data, dataset_vocab, labels_names, max_length):
        self.conll_data = conll_data
        self.vocab = dataset_vocab
        self.pad_token_id = self.vocab["<pad>"]
        self.max_length = int(max_length)
        self.labels_names = labels_names
        self.label2id = {k: ind for ind, k in enumerate(list(self.labels_names))}
        self.id2label = {v: k for k, v in self.label2id.items()}

    def simple_tokenizer(self, sentence):
        return [row[1] for row in sentence if not row[0].startswith("#")]

    def get_labels(self, sentence):
        return [
            self.label2id[row[-1]] for row in sentence if not row[0].startswith("#")
        ]

    def __getitem__(self, idx):
        current_item = self.conll_data[idx]
        input_ids = self.vocab(self.simple_tokenizer(current_item))
        labels = self.get_labels(current_item)
        return {
            "input_ids": input_ids[: self.max_length],
            "labels": labels[: self.max_length],
        }

    def __len__(self):
        return len(self.conll_data)

In [ ]:
# Here we want to figure out empirically max_length

train_n_tokens = pd.Series([len(simple_tokenizer(s)) for s in train_data])

plt.title("Train N Tokens Distribution")
train_n_tokens.hist(bins=100)
plt.show()

q99 = np.quantile(train_n_tokens, 0.99)
print(f"0.99 Quantile of N tokens: {q99}")

In [ ]:
ds_train = TextDataset(train_data, vocab, list(label_dist.keys()), q99)
ds_valid = TextDataset(valid_data, vocab, list(label_dist.keys()), q99)

In [ ]:
class DataCollatorForTokenClassification:
    def __init__(self, pad_token_id, label_pad_token_id=-100):
        self.pad_token_id = pad_token_id
        self.label_pad_token_id = (
            label_pad_token_id  # -100 to ignore tokens in loss computations
        )

    def __call__(self, features):
        # Extract input_ids and labels from the list of feature dictionaries.
        input_ids_list = [f["input_ids"] for f in features]
        labels_list = [f["labels"] for f in features]

        # Determine the maximum sequence length in this batch.
        batch_max_length = max(len(ids) for ids in input_ids_list)

        # Pad input_ids and labels on the right.
        padded_input_ids = [
            ids + [self.pad_token_id] * (batch_max_length - len(ids))
            for ids in input_ids_list
        ]
        padded_labels = [
            labels + [self.label_pad_token_id] * (batch_max_length - len(labels))
            for labels in labels_list
        ]

        # Convert lists to tensors.
        batch_input_ids = torch.tensor(padded_input_ids, dtype=torch.long)
        batch_labels = torch.tensor(padded_labels, dtype=torch.long)

        return {"input_ids": batch_input_ids, "labels": batch_labels}

In [ ]:
pad_token_id = ds_train.vocab["<pad>"]
collator = DataCollatorForTokenClassification(pad_token_id, label_pad_token_id=-100)

dataloader = DataLoader(ds_train, batch_size=64, collate_fn=collator)

In [ ]:
data_iter = iter(dataloader)
first_batch = next(data_iter)
first_batch

## Model


In [ ]:
metric = evaluate.load("seqeval")
label_names = ds_train.labels_names
id2label = ds_train.id2label
label2id = {v: k for k, v in id2label.items()}


def compute_metrics_fn(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    true_labels = [
        [id2label[l] for l in label_row if l != -100] for label_row in labels
    ]
    true_preds = []
    for pred_row, label_row in zip(preds, labels):
        row = []
        for p, l in zip(pred_row, label_row):
            if l != -100:
                row.append(id2label[int(p)])
        true_preds.append(row)

    results = metric.compute(predictions=true_preds, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [ ]:
class UniversalRNNTokenClassifier(nn.Module):
    def __init__(
        self,
        num_embeddings: int,
        num_labels: int,
        embed_dim: int = 256,
        rnn_hidden: int = 512,
        rnn_type: nn.Module = nn.GRU,  # nn.RNN / nn.GRU / nn.LSTM
        n_rnns: int = 3,
        bidirectional: bool = True,
        dropout_rate: float = 0.3,
        pad_id: int = 0,
        average_type: Optional[str] = None,  # kept for API symmetry; unused here
    ):
        super().__init__()
        self.num_labels = num_labels
        self.pad_id = pad_id
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        self.rnn_type = rnn_type
        self.hidden_size = rnn_hidden

        self.embed = nn.Embedding(num_embeddings, embed_dim, padding_idx=pad_id)
        self.dropout = nn.Dropout(dropout_rate)

        self.rnn = rnn_type(
            input_size=embed_dim,
            hidden_size=rnn_hidden,
            num_layers=n_rnns,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout_rate if n_rnns > 1 else 0.0,
        )
        self.classifier = nn.Linear(rnn_hidden * self.num_directions, num_labels)

        self.loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

    def forward(
        self,
        input_ids: torch.LongTensor = None,
        labels: Optional[torch.LongTensor] = None,
        attention_mask: Optional[
            torch.LongTensor
        ] = None,  # not used by RNN; masking is via labels=-100
        **kwargs,
    ):
        # input_ids: [B, T]
        x = self.embed(input_ids)  # [B, T, E]
        x = self.dropout(x)

        # You can pack based on attention_mask or true lengths if you like.
        # For simplicity (and because labels carry -100), we run unpacked:
        out, _ = self.rnn(x)  # [B, T, H*D]
        out = self.dropout(out)
        logits = self.classifier(out)  # [B, T, C]

        loss = None
        if labels is not None:
            # CrossEntropy over time; ignore_index handles padding.
            loss = self.loss_fn(logits.view(-1, self.num_labels), labels.view(-1))

        # Return the standard HF TokenClassifierOutput so Trainer can read loss/logits.
        return TokenClassifierOutput(loss=loss, logits=logits)

In [ ]:
data_collator = DataCollatorForTokenClassification(
    pad_token_id, label_pad_token_id=-100
)


NUM_EMBEDDINGS = len(ds_train.vocab)
NUM_LABELS = len(ds_train.labels_names)

model = UniversalRNNTokenClassifier(
    num_embeddings=NUM_EMBEDDINGS,
    num_labels=NUM_LABELS,
    embed_dim=256,
    rnn_hidden=512,
    rnn_type=nn.GRU,  # nn.LSTM or nn.RNN if needed
    n_rnns=3,
    bidirectional=True,
    dropout_rate=0.3,
    pad_id=pad_token_id,
)

## Train


In [ ]:
wandb.init(project="rnn-lecture-ner", name="gru-bidirectional-3layers", dir=OUT_DIR)

In [ ]:
LEARNING_RATE = 1e-3
BATCH_SIZE = 64
TOTAL_EPOCHS = 5

training_args = TrainingArguments(
    output_dir=str(OUT_DIR / "hf_checkpoints_gru_bi3"),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    num_train_epochs=TOTAL_EPOCHS,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",  # "f1" from compute_metrics
    greater_is_better=True,
    report_to=["wandb"],  # enable W&B
    run_name="gru-bidirectional-3layers",
    gradient_accumulation_steps=1,
    gradient_checkpointing=False,
    dataloader_pin_memory=False,  # not supported on MPS
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_valid,
    data_collator=data_collator,
    compute_metrics=compute_metrics_fn,
)

trainer.train()
eval_metrics = trainer.evaluate()
print("Eval:", eval_metrics)
wandb.finish()

## Inference


In [ ]:
def merge_bio_entities(tokens, bio_tags):
    entities, cur = [], None
    for tok, tag in zip(tokens, bio_tags):
        if tag.startswith("B-"):
            if cur:
                entities.append(cur)
            cur = {"text": tok, "label": tag[2:]}
        elif tag.startswith("I-") and cur and cur["label"] == tag[2:]:
            cur["text"] += " " + tok
        else:  # "O" or mismatched I-
            if cur:
                entities.append(cur)
                cur = None
    if cur:
        entities.append(cur)
    return entities


def _infer(self, text, vocab, id2label, tokenizer=None):
    device = next(self.parameters()).device
    tok = (tokenizer or word_tokenize)(text)
    ids = vocab(tok)  # expects: vocab(List[str]) -> List[int]
    inp = torch.tensor([ids], dtype=torch.long, device=device)  # [1, T]

    self.eval()
    with torch.no_grad():
        out = self(input_ids=inp)  # TokenClassifierOutput
        logits = out.logits  # [1, T, C]
        pred_ids = logits.argmax(dim=-1).squeeze(0).tolist()

    bio_tags = [id2label[p] for p in pred_ids]
    return merge_bio_entities(tok, bio_tags)


best_model = trainer.model.eval()

# attach method to the model class or just to this instance
best_model.infer = types.MethodType(_infer, best_model)

In [ ]:
best_model.infer("Привіт, як у тебе справи?", vocab, id2label)

In [ ]:
text = """Палата відділення щелепнолицевої хірургії по вулиці Зоологічній,
в палаті на кроватях лежать Василь Гнатович і Енгельс Гасанович"""
best_model.infer(text, vocab, id2label)

In [ ]:
vocab["Зоологічній"], vocab["Василь"], vocab["Гнатович"], vocab["Енгельс"], vocab[
    "Гасанович"
]

In [ ]:
text = """Недалеко от Богуслава, коло Росі, в довгому покрученому яру розкинулось село Семигори.
Яр в'ється гадюкою між крутими горами, між зеленими терасами;
од яру на всі боки розбіглись, неначе гілки дерева, глибокі рукави й поховались десь далеко в густих лісах."""
best_model.infer(text, vocab, id2label)